# Logistic Regression (Baseline)

**Goal:** build a simple, interpretable baseline for predicting hotel booking cancellations. The more complex models (XGBoost, MLP) will be compared against it.

**Data:** the preprocessed bookings (`src/preprocessing.py`), split randomly into train (70%), validation (15%) and test (15%), stratified by the target. Settings are chosen on the validation set; the test set is used only once, for the final model.

**Metrics:** F1-score and ROC-AUC, since the classes are imbalanced (27.6% canceled) and accuracy would be misleading.

In [11]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent                                  
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_split import main as create_data_splits, split_features_target      
from src.evaluation import evaluate                                

In [12]:
splits = create_data_splits()

X_train, y_train = split_features_target(splits["random"]["train"])         
X_val, y_val = split_features_target(splits["random"]["validation"])
X_test, y_test = split_features_target(splits["random"]["test"])

print(X_train.shape, X_val.shape, X_test.shape)

(59576, 29) (12766, 29) (12767, 29)


In [13]:
numerical_columns = X_train.select_dtypes(include="number").columns.tolist()   
categorical_columns = X_train.select_dtypes(exclude="number").columns.tolist()  

print(len(numerical_columns), "numerical:", numerical_columns)
print(len(categorical_columns), "categorical:", categorical_columns)

16 numerical: ['lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'booking_changes', 'days_in_waiting_list', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'total_nights', 'total_guests']
13 categorical: ['hotel', 'arrival_date_month', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'assigned_room_type', 'deposit_type', 'agent', 'company', 'customer_type', 'arrival_weekday']


## Encoding and scaling

- **Numerical features:** standardized with `StandardScaler`, because Logistic Regression is sensitive to feature scale.
- **Categorical features:** one-hot encoded. Categories that appear fewer than 50 times in the training set are grouped into a single "infrequent" column (`min_frequency=50`). Without grouping, `country`, `agent` and `company` alone would create almost 800 mostly empty columns.

Both steps are placed inside a `Pipeline`, so they are fitted on the training set only and then applied to the validation and test sets. This prevents information from those sets leaking into training.

In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_columns),                                   
    ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=50), categorical_columns), 
])

## Majority-class baseline

A model that predicts "not canceled" for every booking. It learns nothing from the features and sets the lower bound that any useful model must beat.

In [15]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy="most_frequent")                         
dummy.fit(X_train, y_train)                                            

y_prob_dummy = dummy.predict_proba(X_val)[:, 1]                  
evaluate("Majority class", y_val, y_prob_dummy, "results_buse.csv", split="validation")   

Majority class (validation): F1=0.000, ROC-AUC=0.500


/Users/busebaylav/Desktop/GitHub/hotel-cancellation/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'model': 'Majority class',
 'split': 'validation',
 'f1': 0.0,
 'roc_auc': 0.5,
 'precision': 0.0,
 'recall': 0.0}

In [16]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

logreg = Pipeline([
    ("preprocessor", preprocessor),                                      
    ("model", LogisticRegression(max_iter=1000)),
])
logreg.fit(X_train, y_train)

y_prob_logreg = logreg.predict_proba(X_val)[:, 1]                        
evaluate("Logistic Regression", y_val, y_prob_logreg, "results_buse.csv", split="validation")

Logistic Regression (validation): F1=0.604, ROC-AUC=0.862


{'model': 'Logistic Regression',
 'split': 'validation',
 'f1': 0.6038095238095238,
 'roc_auc': 0.8616432626415887,
 'precision': 0.6854054054054054,
 'recall': 0.5395744680851063}

**Interpretation:** Logistic Regression clearly beats the majority-class baseline (ROC-AUC 0.862 vs 0.5), so it separates canceled from non-canceled bookings well. However, recall is only 0.54: the model misses almost half of the cancellations. Because only 27.6% of bookings are canceled, the predicted probabilities lean towards "not canceled", and many cancellations fall below the 0.5 threshold.

In [17]:
for class_weight in [None, "balanced"]:                                   # (1)
    for C in [0.01, 0.1, 1, 10]:
        model = Pipeline([
            ("preprocessor", preprocessor),
            ("model", LogisticRegression(max_iter=2000, C=C, class_weight=class_weight)),   # (2)
        ])
        model.fit(X_train, y_train)                                          # (3)
        y_prob = model.predict_proba(X_val)[:, 1]                      # (4)
        evaluate(f"LogReg C={C} cw={class_weight}", y_val, y_prob,
                 "results_buse.csv", split="validation")

LogReg C=0.01 cw=None (validation): F1=0.589, ROC-AUC=0.859
LogReg C=0.1 cw=None (validation): F1=0.603, ROC-AUC=0.862
LogReg C=1 cw=None (validation): F1=0.604, ROC-AUC=0.862
LogReg C=10 cw=None (validation): F1=0.605, ROC-AUC=0.861
LogReg C=0.01 cw=balanced (validation): F1=0.662, ROC-AUC=0.860
LogReg C=0.1 cw=balanced (validation): F1=0.663, ROC-AUC=0.862
LogReg C=1 cw=balanced (validation): F1=0.665, ROC-AUC=0.862
LogReg C=10 cw=balanced (validation): F1=0.665, ROC-AUC=0.862


**Interpretation:**
- The regularization strength `C` has almost no effect (F1 0.59–0.61). With about 59,000 training rows and 207 features, the model is not overfitting.
- `class_weight="balanced"`, which gives more weight to the minority class, raises F1 from 0.604 to 0.665: recall increases from 0.54 to 0.82, while precision decreases from 0.69 to 0.56.
- ROC-AUC does not change (0.862). Class weighting does not change how the model ranks bookings; it shifts the predicted probabilities, which has the same effect as lowering the decision threshold. This confirms that the model's ranking was already good and the low recall was caused by the threshold.

We select `C=1` and `class_weight="balanced"`.

In [18]:
best_logreg = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000, C=1, class_weight="balanced")),   
])
best_logreg.fit(X_train, y_train)

y_prob_test = best_logreg.predict_proba(X_test)[:, 1]                       
evaluate("Logistic Regression", y_test, y_prob_test, "results_buse.csv", split="random_test")   

Logistic Regression (random_test): F1=0.660, ROC-AUC=0.860


{'model': 'Logistic Regression',
 'split': 'random_test',
 'f1': 0.660351201478743,
 'roc_auc': 0.8604047970949764,
 'precision': 0.5570064314948353,
 'recall': 0.8107801418439716}

## Results

| Model | Set | F1 | ROC-AUC | Precision | Recall |
|---|---|---|---|---|---|
| Majority class | Validation | 0.000 | 0.500 | 0.000 | 0.000 |
| Logistic Regression (default) | Validation | 0.604 | 0.862 | 0.685 | 0.540 |
| Logistic Regression (balanced, C=1) | Validation | 0.665 | 0.862 | 0.559 | 0.820 |
| **Logistic Regression (balanced, C=1)** | **Test** | **0.660** | **0.860** | **0.557** | **0.811** |

The test scores are very close to the validation scores, so the model generalizes well to unseen bookings. This is the baseline for the other models.

**Trade-off:** the balanced model catches about 81% of cancellations at the cost of more false alarms (precision 0.56). Which side matters more depends on the hotel: a missed cancellation means an empty room, a false alarm may lead to unnecessary overbooking.

**Next steps (for all models):** temporal test (train on 2015–2016, test on 2017), ablation of the suspected leakage features, and feature importance analysis.